# Ejecución batch del pipeline Spike–ADR

Este notebook importa `run_batch_by_folder.py` y ejecuta el pipeline sobre todos los PDFs ubicados en una carpeta.

La lógica esperada es:

```text
input_dir/*.pdf
        ↓
run_folder_batch(...)
        ↓
output_dir/<nombre_del_pdf>/
        ├── result.json
        ├── signals.json
        ├── SPIKES_ADR.md
        ├── evaluation_rubric.md
        ├── evaluation_data.csv
        ├── graph.dot
        ├── graph.png
        ├── report.tex
        └── report.pdf  # opcional
```

Antes de ejecutar, asegúrate de que este notebook esté en la misma carpeta que `pipeline.py`, `run_batch_by_folder.py`, `models.py`, etc., o ajusta `PIPELINE_DIR`.

In [ ]:
# Celda 1 — Configuración de rutas
from pathlib import Path
import sys
import os
import json
import pandas as pd
from IPython.display import display, Markdown

# Carpeta donde está el código fuente del pipeline.
# Si este notebook está dentro de spike_adr_pipeline_v2_modified, deja '.'.
PIPELINE_DIR = Path('.').resolve()

# Carpeta que contiene los PDFs a analizar.
INPUT_DIR = Path('./pdfs').resolve()

# Carpeta raíz donde se generarán los resultados.
OUTPUT_DIR = Path('./results_batch').resolve()

# Opciones de ejecución
RECURSIVE = False       # True si quieres buscar PDFs en subcarpetas
COMPILE_PDF = False     # True si tienes pdflatex instalado y quieres report.pdf
GENERATE_LATEX = True   # Genera report.tex
LIMIT = None            # Usa un número, por ejemplo 2, para probar solo los primeros PDFs

# Agregar el pipeline al path de Python
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

print('PIPELINE_DIR =', PIPELINE_DIR)
print('INPUT_DIR    =', INPUT_DIR)
print('OUTPUT_DIR   =', OUTPUT_DIR)

## Verificación previa

Esta celda revisa que existan los archivos fuente necesarios y que haya PDFs en la carpeta de entrada.

In [ ]:
# Celda 2 — Verificación previa
required_files = [
    'run_batch_by_folder.py',
    'pipeline.py',
    'models.py',
    'pdf_parser.py',
    'signal_extractor.py',
    'spike_generator.py',
    'adr_generator.py',
    'graph_builder.py',
    'evaluate.py',
    'latex_report.py',
]

missing = [f for f in required_files if not (PIPELINE_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        'Faltan archivos del pipeline en PIPELINE_DIR:\n' + '\n'.join(missing)
    )

if not INPUT_DIR.exists():
    raise FileNotFoundError(f'No existe INPUT_DIR: {INPUT_DIR}')

pattern = '**/*.pdf' if RECURSIVE else '*.pdf'
pdfs = sorted(INPUT_DIR.glob(pattern))
if LIMIT is not None:
    pdfs = pdfs[:LIMIT]

print(f'PDFs encontrados: {len(pdfs)}')
for p in pdfs:
    print('-', p.name)

if not pdfs:
    raise FileNotFoundError('No se encontraron PDFs para procesar.')

## Importar el batch runner

Importamos `run_folder_batch`, que internamente llama a `run_pipeline()` una vez por cada PDF.

In [ ]:
# Celda 3 — Importar módulo batch
import importlib
import run_batch_by_folder
importlib.reload(run_batch_by_folder)

from run_batch_by_folder import run_folder_batch

print('Módulo importado correctamente:', run_batch_by_folder.__file__)

## Ejecutar batch con progreso visible

El progreso se verá en la salida de la celda porque `run_folder_batch()` imprime el avance `[i/N]`, el nombre del PDF, la carpeta de salida y el resumen final.

In [ ]:
# Celda 4 — Ejecutar batch
statuses = run_folder_batch(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    recursive=RECURSIVE,
    compile_pdf=COMPILE_PDF,
    generate_latex=GENERATE_LATEX,
    continue_on_error=True,
    limit=LIMIT,
)

print('\nObjetos de estado generados:', len(statuses))

## Ver resumen en tabla

Esta celda carga `batch_summary.csv` y muestra el estado de cada proyecto.

In [ ]:
# Celda 5 — Tabla resumen
summary_csv = OUTPUT_DIR / 'batch_summary.csv'
summary_json = OUTPUT_DIR / 'batch_summary.json'

if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    display(df)
else:
    print('No se encontró:', summary_csv)

if summary_json.exists():
    data = json.loads(summary_json.read_text(encoding='utf-8'))
    display(Markdown(
        f"""
### Resumen global

- Proyectos totales: **{data.get('total_projects', 0)}**
- Proyectos OK: **{data.get('ok_projects', 0)}**
- Proyectos con error: **{data.get('error_projects', 0)}**
- Señales totales: **{data.get('total_signals', 0)}**
- Spikes totales: **{data.get('total_spikes', 0)}**
- ADRs totales: **{data.get('total_adrs', 0)}**
        """
    ))

## Revisar archivos generados

Esta celda lista las carpetas generadas por proyecto y muestra los artefactos principales.

In [ ]:
# Celda 6 — Listar artefactos por proyecto
project_dirs = sorted([p for p in OUTPUT_DIR.iterdir() if p.is_dir()]) if OUTPUT_DIR.exists() else []

for project_dir in project_dirs:
    print('\n' + '=' * 90)
    print(project_dir.name)
    print('=' * 90)
    for artifact in [
        'result.json',
        'signals.json',
        'SPIKES_ADR.md',
        'evaluation_rubric.md',
        'evaluation_data.csv',
        'graph.dot',
        'graph.png',
        'report.tex',
        'report.pdf',
        'error.log',
    ]:
        path = project_dir / artifact
        if path.exists():
            size_kb = path.stat().st_size / 1024
            print(f'- {artifact:24s} {size_kb:8.1f} KB')

## Vista rápida de un resultado Markdown

Selecciona un índice de proyecto y muestra las primeras líneas de `SPIKES_ADR.md`.

In [ ]:
# Celda 7 — Vista rápida del primer SPIKES_ADR.md
PROJECT_INDEX = 0

if project_dirs:
    selected_dir = project_dirs[PROJECT_INDEX]
    md_path = selected_dir / 'SPIKES_ADR.md'
    if md_path.exists():
        text = md_path.read_text(encoding='utf-8')
        preview = '\n'.join(text.splitlines()[:80])
        display(Markdown(preview))
    else:
        print('No existe SPIKES_ADR.md en:', selected_dir)
else:
    print('No hay carpetas de proyecto generadas.')

## Reejecución limpia opcional

Usa esta celda solo si quieres borrar completamente `OUTPUT_DIR` y volver a ejecutar desde cero.

In [ ]:
# Celda 8 — Limpieza opcional
# Descomenta con cuidado si quieres borrar resultados previos.

# import shutil
# if OUTPUT_DIR.exists():
#     shutil.rmtree(OUTPUT_DIR)
#     print('Carpeta eliminada:', OUTPUT_DIR)
# else:
#     print('No existe:', OUTPUT_DIR)